# Gradeformer — Part 2: Third external cohort, multi-seed ensemble, interpretability

**Run this AFTER Part 1 has been published as a Kaggle dataset.**

Reuses Part 1's checkpoints/embeddings/graphs (no re-extraction, no re-training of
the patch classifiers) for:
1. **Gleason2019 held-out core-level external validation** (full ~20% split -- this
   notebook has its own separate time budget, so no reduction is needed here)
2. **Multi-seed / architecture-diverse graph-training ensemble** (full config: 3
   seeds x 5-fold x 30 epochs, matching Part 1's own Stage 13 settings)
3. **Interpretability analyses**: attention-to-annotation concordance, embedding
   t-SNE/UMAP, fold-ensemble uncertainty vs error, adjacent-vs-major error breakdown

**Before running, attach TWO inputs:**
- Your published Part-1 output dataset (e.g. `aminurjoyy/gradeformer`)
- The original Gleason2019 dataset (`danielerussica/gleason2019-grand-challenge`)

Enable Internet + GPU. `HF_TOKEN` secret needed only if the UNI backbone has to be
rebuilt (it does, for Gleason2019 embedding extraction).


*Gleason2019 path is auto-detected (`/kaggle/input/datasets/danielerussica/gleason2019-grand-challenge`, nested folders handled).*

In [1]:

import subprocess, sys
def _pip(*a):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
    except Exception as e:
        print("pip warning:", e)
try:
    import torch_geometric
except Exception:
    _pip("--no-deps", "torch-geometric")
    for dep in ["fsspec", "aiohttp", "psutil"]:
        try: __import__(dep)
        except Exception: _pip(dep)
    import torch_geometric
print("torch_geometric:", torch_geometric.__version__)
try:
    import umap
except Exception:
    _pip("umap-learn")
    import umap

import os, re, math, hashlib, random, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import torchvision.transforms as T
from sklearn.metrics import cohen_kappa_score, accuracy_score
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.manifold import TSNE

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def find_dataset_root(*candidates, marker="embeddings"):
    """Try each candidate path; if none match directly, search one level deep for
    a folder containing `marker` (handles Kaggle nesting the dataset an extra level)."""
    for c in candidates:
        p = Path(c)
        if p.exists():
            for sub in p.rglob(marker):
                if sub.is_dir():
                    return sub.parent
            return p   # exists but marker not found -- return as-is, caller checks further
    for c in candidates:
        p = Path(c)
        if p.parent.exists():
            hits = list(p.parent.rglob(marker))
            if hits:
                return hits[0].parent
    raise FileNotFoundError(f"None of {candidates} found, and no '{marker}' folder nearby.")

INPUT_ROOT = find_dataset_root(
    "/kaggle/input/datasets/aminurjoyy/gradeformer",
    "/kaggle/input/gradeformer",
    marker="embeddings")
print("Part-1 output dataset (INPUT_ROOT):", INPUT_ROOT)
print("Contents:", sorted(p.name for p in INPUT_ROOT.iterdir())[:30])

# ---- Gleason2019 raw dataset: robust auto-discovery (path + nested folders) ----
GLEASON_BASE = Path("/kaggle/input/datasets/danielerussica/gleason2019-grand-challenge")
IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

def _n_imgs(d):
    try: return sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXT)
    except Exception: return 0

def find_gleason_dirs():
    cands = [GLEASON_BASE, Path("/kaggle/input/gleason2019-grand-challenge")]
    roots = [c for c in cands if c.exists()]
    if not roots:   # last resort: any folder under /kaggle/input with 'gleason' in its name
        roots = [p for p in Path("/kaggle/input").rglob("*") if p.is_dir() and "gleason" in p.name.lower()][:1]
    if not roots:
        return None, None, None
    root = roots[0]
    dirs = [root] + [p for p in root.rglob("*") if p.is_dir()]
    # masks: folder named like STAPLED_maps (fallback: any folder with 'map')
    mdirs = [d for d in dirs if "stapled" in d.name.lower() and _n_imgs(d) > 0]
    if not mdirs:
        mdirs = [d for d in dirs if "map" in d.name.lower() and _n_imgs(d) > 0]
    mask_dir = max(mdirs, key=_n_imgs) if mdirs else None
    # images: folder named like 'Train Imgs' (fallback: folder with most images that is not a mask folder)
    idirs = [d for d in dirs if "train" in d.name.lower() and "img" in d.name.lower() and _n_imgs(d) > 0]
    if not idirs:
        idirs = [d for d in dirs if "map" not in d.name.lower() and "stapled" not in d.name.lower()
                 and "test" not in d.name.lower() and _n_imgs(d) > 0]
    img_dir = max(idirs, key=_n_imgs) if idirs else None
    return root, img_dir, mask_dir

GLEASON_ROOT, GLEASON_IMG, GLEASON_MASK = find_gleason_dirs()
if GLEASON_ROOT is None:
    print("WARNING: Gleason2019 raw dataset not found under /kaggle/input -- attach "
          "'danielerussica/gleason2019-grand-challenge'. Part A will be skipped.")
else:
    print("GLEASON_ROOT:", GLEASON_ROOT)
    print("  image dir :", GLEASON_IMG, "->", _n_imgs(GLEASON_IMG) if GLEASON_IMG else 0, "files")
    print("  mask dir  :", GLEASON_MASK, "->", _n_imgs(GLEASON_MASK) if GLEASON_MASK else 0, "files")
    if GLEASON_IMG is None or GLEASON_MASK is None:
        print("  !! could not locate image/mask folders; folder tree (2 levels):")
        for p in sorted(GLEASON_ROOT.iterdir())[:20]:
            print("    ", p.name, "/" if p.is_dir() else "")
            if p.is_dir():
                for q in sorted(p.iterdir())[:8]: print("        ", q.name)

WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
FIG_DIR = WORK / "analysis_figures"; FIG_DIR.mkdir(exist_ok=True)

# ---- config matching the UNI run that produced INPUT_ROOT's artefacts ----
ENCODER = "uni"
EMBED_DIM, INPUT_SIZE = 1024, 224
NODE_FEAT_DIM = EMBED_DIM + 3
HIDDEN_DIM, NUM_HEADS, NUM_LAYERS, ILRA_LATENT, DROPOUT = 512, 8, 3, 128, 0.1
NUM_ISUP, GRAPH_OBJECTIVE, N_BOOTSTRAP = 6, "ordinal", 4000
BATCH_SIZE_GRAPH = 4
EPOCHS_GRAPH, PATIENCE_GRAPH, N_FOLDS = 30, 5, 5     # FULL config (own time budget now)
LR_FINETUNE, WEIGHT_DECAY = 1e-4, 2e-3
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
PATCH_SIZE, GLEASON_TILE, TISSUE_THRESHOLD, KNN_K = 512, 512, 0.05, 8
GLEASON_MASK_MAP = {0: "Benign", 3: "GP3", 4: "GP4", 5: "GP5"}
ISUP_MAP = {("GP3", "GP3"): 1, ("GP3", "GP4"): 2, ("GP4", "GP3"): 2,
            ("GP4", "GP4"): 3, ("GP4", "GP5"): 4, ("GP5", "GP4"): 4,
            ("GP3", "GP5"): 3, ("GP5", "GP3"): 3, ("GP5", "GP5"): 5}
SICAP_VAL_LABELS = INPUT_ROOT / "sicap_val_labels.csv"
SICAP_SLIDE_VAL  = INPUT_ROOT / "sicap_slide_labels_val.csv"
SICAP_SLIDE_TRAIN = INPUT_ROOT / "sicap_slide_labels_train.csv"
PANDA_SLIDE_LABELS = INPUT_ROOT / "panda_slide_labels.csv"
OOF_PATH = INPUT_ROOT / "oof_calibration.csv"
print("Config ready.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00
torch_geometric: 2.8.0.post1
Device: cuda
Part-1 output dataset (INPUT_ROOT): /kaggle/input/datasets/aminurjoyy/gradeformer/_output_
Contents: ['ablation_results.csv', 'backbone_comparison.csv', 'binary_classifier_best.pt', 'binary_classifier_final.pt', 'embeddings', 'figures', 'final_results_summary.json', 'gleason_tiles', 'gp_classifier_best.pt', 'graph_model_fold0.pt', 'graph_model_fold1.pt', 'graph_model_fold2.pt', 'graph_model_fold3.pt', 'graph_model_fold4.pt', 'graphs', 'heatmaps', 'oof_calibration.csv', 'panda_auroc_per_class.png', 'panda_confusion_matrix.png', 'panda_provider_wise_qwk.csv', 'panda_slide_labels.csv', 'panda_tta_comparison.csv', 'sicap_all_labels.csv', 'sicap_gp_train.csv', 'sicap_gp_val.csv', 'sicap_multiscale', 'sicap_norm', 'sicap_slide_labels_train.csv', 'sicap_slide_labels_val.csv', 'sicap_train_labels.c

## Model architecture + shared helpers (identical to Part 1)

In [2]:

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoLoader
from torch_geometric.nn import TransformerConv, global_mean_pool
from torch_geometric.utils import to_dense_batch, dropout_edge
try:
    import torch_cluster  # noqa: F401
    from torch_geometric.nn import knn_graph
    _HAVE_KNN = True
except Exception:
    _HAVE_KNN = False

class GraphTransformerLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=DROPOUT):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.conv = TransformerConv(hidden_dim, hidden_dim // num_heads, heads=num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(hidden_dim * 4, hidden_dim))
    def forward(self, x, edge_index):
        x = x + self.conv(self.norm1(x), edge_index)
        x = x + self.ffn(self.norm2(x))
        return x

class ILRAPooling(nn.Module):
    def __init__(self, hidden_dim, num_latent=ILRA_LATENT, num_heads=4, dropout=DROPOUT):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(num_latent, hidden_dim) * 0.02)
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.last_attn = None
    def forward(self, x, batch):
        dense, mask = to_dense_batch(x, batch)
        B = dense.shape[0]
        q = self.latents.unsqueeze(0).expand(B, -1, -1)
        out, attn = self.attn(q, dense, dense, key_padding_mask=~mask, need_weights=True)
        self.last_attn = attn.detach()
        return self.norm(out).mean(1)

class GraphTransformerILRA(nn.Module):
    def __init__(self, node_feat_dim=NODE_FEAT_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
                 num_heads=NUM_HEADS, ilra_latent=ILRA_LATENT, dropout=DROPOUT,
                 num_classes=1, pool="ilra"):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(node_feat_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.layers = nn.ModuleList([GraphTransformerLayer(hidden_dim, num_heads, dropout)
                                     for _ in range(num_layers)])
        self.pool_type = pool
        self.ilra = ILRAPooling(hidden_dim, ilra_latent, 4, dropout) if pool != "mean" else None
        self.cls = nn.Sequential(nn.Linear(hidden_dim, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, num_classes))
    def forward(self, data):
        x = self.proj(data.x)
        for l in self.layers:
            x = l(x, data.edge_index)
        emb = global_mean_pool(x, data.batch) if self.pool_type == "mean" else self.ilra(x, data.batch)
        return self.cls(emb)

def graph_out_dim(): return 1 if GRAPH_OBJECTIVE == "ordinal" else NUM_ISUP
def graph_pred(out):
    return out.squeeze(-1).round().clamp(0, NUM_ISUP - 1).long() if GRAPH_OBJECTIVE == "ordinal" else out.argmax(1)

def _knn_edges(coords, k):
    N = coords.shape[0]; k = min(k, max(1, N - 1))
    if N <= 1:
        return torch.zeros((2, 0), dtype=torch.long)
    if _HAVE_KNN:
        return knn_graph(torch.tensor(coords, dtype=torch.float), k=k, loop=False)
    d = np.linalg.norm(coords[:, None] - coords[None], axis=-1); np.fill_diagonal(d, np.inf)
    nn_idx = np.argsort(d, axis=1)[:, :k]
    src = np.repeat(np.arange(N), k); dst = nn_idx.reshape(-1)
    return torch.tensor(np.vstack([src, dst]), dtype=torch.long)

class SlideGraphDataset(torch.utils.data.Dataset):
    def __init__(self, root, augment=False, ids=None):
        self.files = sorted(Path(root).glob("*.pt"))
        if ids is not None:
            ids = set(ids); self.files = [f for f in self.files if f.stem in ids]
        self.augment = augment
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        data = torch.load(self.files[i], weights_only=False)
        if self.augment:
            data = data.clone()
            N = data.x.shape[0]
            keep = torch.rand(N) > 0.10
            if keep.sum() >= 4:
                idx = torch.nonzero(keep).squeeze(-1)
                remap = -torch.ones(N, dtype=torch.long); remap[idx] = torch.arange(len(idx))
                mask_e = keep[data.edge_index[0]] & keep[data.edge_index[1]]
                data.edge_index = remap[data.edge_index[:, mask_e]]
                data.x = data.x[idx]
            data.x = data.x.clone(); data.x[:, -2:] += torch.randn_like(data.x[:, -2:]) * 0.01
            if data.edge_index.shape[1] > 0:
                data.edge_index, _ = dropout_edge(data.edge_index, p=0.10)
        return data

def graph_loader(root, augment=False, shuffle=False, ids=None, bs=BATCH_SIZE_GRAPH):
    return GeoLoader(SlideGraphDataset(root, augment, ids), batch_size=bs, shuffle=shuffle, num_workers=0)

def bootstrap_qwk(y_true, y_pred, n=N_BOOTSTRAP, seed=0):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    if len(y_true) == 0:
        return 0.0, (0.0, 0.0)
    base = cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=list(range(6)))
    rng = np.random.RandomState(seed); stats = []
    for _ in range(n):
        idx = rng.randint(0, len(y_true), len(y_true))
        if len(set(y_true[idx])) < 2: continue
        stats.append(cohen_kappa_score(y_true[idx], y_pred[idx], weights="quadratic", labels=list(range(6))))
    if not stats: return base, (base, base)
    return base, (float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5)))

class OptimizedRounder:
    def __init__(self, n=NUM_ISUP): self.coef = np.arange(n - 1) + 0.5
    def _qwk(self, coef, X, y):
        return -cohen_kappa_score(y, np.digitize(X, np.sort(coef)), weights="quadratic", labels=list(range(NUM_ISUP)))
    def fit(self, X, y):
        import scipy.optimize as spo
        try:
            r = spo.minimize(self._qwk, self.coef, args=(X, y), method="Nelder-Mead")
            self.coef = np.sort(r.x)
        except Exception as e:
            print("  calibration failed:", e)
        return self
    def predict(self, X): return np.digitize(X, self.coef)

def patient_id(slide_id):
    s = str(slide_id); return s.rsplit("_", 1)[0] if "_" in s else s
def get_slide_id(filename): return Path(filename).stem.split("_")[0]
def get_coords(filename):
    stem = Path(filename).stem
    m = re.search(r"xini[_]?(\d+).*?yini[_]?(\d+)", stem)
    if m:
        x, y = int(m.group(1)), int(m.group(2)); return (y // PATCH_SIZE, x // PATCH_SIZE)
    nums = re.findall(r"\d+", stem)
    if len(nums) >= 2: return (int(nums[-2]), int(nums[-1]))
    return (0, 0)

def load_fold_ensemble():
    models = []
    for cp in sorted(INPUT_ROOT.glob("graph_model_fold*.pt")) or sorted(INPUT_ROOT.rglob("graph_model_fold*.pt")):
        m = GraphTransformerILRA(num_classes=graph_out_dim()).to(DEVICE)
        try:
            m.load_state_dict(torch.load(cp, map_location=DEVICE)); m.eval(); models.append(m)
        except Exception as e:
            print("  skip", cp.name, e)
    print(f"Loaded {len(models)} Part-1 fold checkpoint(s).")
    return models

PANDA_GRAPH_DIR = INPUT_ROOT / "graphs" / "panda"
SICAP_TRAIN_GRAPH_DIR = INPUT_ROOT / "graphs" / "sicap" / "train"
SICAP_VAL_GRAPH_DIR = INPUT_ROOT / "graphs" / "sicap" / "val"
panda_gl = graph_loader(PANDA_GRAPH_DIR) if PANDA_GRAPH_DIR.exists() else None
ENS = load_fold_ensemble()
print("Helpers ready. PANDA graphs found:", PANDA_GRAPH_DIR.exists())


Loaded 5 Part-1 fold checkpoint(s).
Helpers ready. PANDA graphs found: True


## Part A — Gleason2019 held-out core-level external validation (full ~20% split)

Needs the UNI backbone rebuilt (gated -> HF_TOKEN) to embed held-out cores.

In [3]:

HF_TOKEN = ""   # optional: paste 'hf_...' here, or enable Kaggle secret HF_TOKEN for THIS notebook
def _get_hf_token():
    tok = HF_TOKEN or os.environ.get("HF_TOKEN", "")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = ""
    return tok

def _hf_login():
    tok = _get_hf_token()
    if not tok:
        raise RuntimeError(
            "HF_TOKEN not found. In this notebook: Add-ons -> Secrets -> add HF_TOKEN and tick the "
            "'attached' checkbox for THIS notebook (secrets are per-notebook), or paste the token in HF_TOKEN above. "
            "The HF account must also have been granted access to MahmoodLab/UNI.")
    from huggingface_hub import login
    login(token=tok)
    return True

class FeatWrap(nn.Module):
    def __init__(self, backbone): super().__init__(); self.backbone = backbone
    def forward(self, x): return self.backbone(x)

def build_uni_feat():
    import timm; _hf_login()
    bb = timm.create_model("hf-hub:MahmoodLab/UNI", pretrained=True,
                           init_values=1e-5, dynamic_img_size=True)
    return FeatWrap(bb).to(DEVICE).eval()

def gleason_core_label(mask_path):
    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    counts = {name: int((m == v).sum()) for v, name in GLEASON_MASK_MAP.items()}
    total = sum(counts.values())
    if total == 0: return None
    pct = {k: 100 * v / total for k, v in counts.items() if k != "Benign"}
    present = {k: v for k, v in pct.items() if v > 5}
    if not present: return 0
    ranked = sorted(present, key=present.get, reverse=True)
    pr = ranked[0]; se = ranked[1] if len(ranked) > 1 else pr
    return ISUP_MAP.get((pr, se), ISUP_MAP.get((pr, pr), 1))

def build_mask_index(mask_dir):
    return {f: mask_dir / f for f in os.listdir(mask_dir) if f.lower().endswith(IMG_EXT)}

def find_mask(stem, mask_index):
    for c in (stem + ".png", stem + "_mask.png", stem + "_classimg_nonconvex.png"):
        if c in mask_index: return mask_index[c]
    for name, p in mask_index.items():          # any mask file that starts with the core stem
        if name.startswith(stem + "_") or name.startswith(stem + "."):
            return p
    return None

def is_held_out(stem, frac=0.20):
    h = int(hashlib.md5(stem.encode()).hexdigest(), 16)
    return (h % 100) < int(frac * 100)

_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

@torch.no_grad()
def extract_gleason_core(FEAT, img_path, core_id, out_dir):
    img = cv2.imread(str(img_path))
    if img is None: return False
    H, W = img.shape[:2]
    if H < GLEASON_TILE or W < GLEASON_TILE: return False
    e10b, e20b, coords = [], [], []
    for yy in range(0, H - GLEASON_TILE, GLEASON_TILE):
        for xx in range(0, W - GLEASON_TILE, GLEASON_TILE):
            tile = img[yy:yy + GLEASON_TILE, xx:xx + GLEASON_TILE]
            gray = cv2.cvtColor(tile, cv2.COLOR_BGR2GRAY)
            _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
            if float((th > 0).mean()) < TISSUE_THRESHOLD:
                continue
            tile_rgb = cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)
            im = Image.fromarray(tile_rgb)
            t10 = im.resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC)
            c = (GLEASON_TILE - 256) // 2
            t20 = im.crop((c, c, c + 256, c + 256)).resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC)
            e10b.append(_tf(t10)); e20b.append(_tf(t20))
            coords.append([yy // GLEASON_TILE, xx // GLEASON_TILE])
    if not e10b: return False
    with torch.cuda.amp.autocast():
        f10 = FEAT(torch.stack(e10b).to(DEVICE)).float().cpu().numpy()
        f20 = FEAT(torch.stack(e20b).to(DEVICE)).float().cpu().numpy()
    np.save(out_dir / f"{core_id}_10x.npy", f10); np.save(out_dir / f"{core_id}_20x.npy", f20)
    np.save(out_dir / f"{core_id}_coords.npy", np.array(coords, dtype=np.float32))
    return True

def build_gleason_graph(core_id, isup_grade, emb_dir):
    e10 = np.load(emb_dir / f"{core_id}_10x.npy"); e20 = np.load(emb_dir / f"{core_id}_20x.npy")
    coords = np.load(emb_dir / f"{core_id}_coords.npy")
    N = min(len(e10), len(e20), len(coords))
    if N == 0: return None
    e10, e20, coords = e10[:N], e20[:N], coords[:N]
    cn = coords - coords.mean(0, keepdims=True); cn = cn / (np.abs(cn).max() + 1e-6)
    x10 = np.concatenate([e10, np.zeros((N, 1)), cn], 1); x20 = np.concatenate([e20, np.ones((N, 1)), cn], 1)
    x = torch.tensor(np.concatenate([x10, x20], 0), dtype=torch.float)
    e_intra10 = _knn_edges(coords, KNN_K); e_intra20 = _knn_edges(coords, KNN_K) + N
    cross = torch.stack([torch.arange(N), torch.arange(N) + N]); cross = torch.cat([cross, cross.flip(0)], 1)
    edge_index = torch.cat([e_intra10, e_intra20, cross], 1)
    return Data(x=x, edge_index=edge_index, y=torch.tensor([int(isup_grade)], dtype=torch.long), slide_id=core_id)

def run_part_a():
    global FEAT
    if GLEASON_IMG is None or GLEASON_MASK is None:
        print("SKIP Part A: Gleason2019 image/mask folders not found (see discovery printout above)."); return
    img_files = sorted(f for f in os.listdir(GLEASON_IMG) if f.lower().endswith(IMG_EXT))
    held_files = [f for f in img_files if is_held_out(Path(f).stem, frac=0.20)]
    print(f"Gleason2019 total cores: {len(img_files)} | held-out (full ~20% split): {len(held_files)}")
    if not held_files:
        print("SKIP Part A: no images found in", GLEASON_IMG); return
    mask_index = build_mask_index(GLEASON_MASK)
    print(f"Mask files indexed: {len(mask_index)} | example image: {img_files[0]} | example mask: {next(iter(mask_index), None)}")
    gl_emb = WORK / "embeddings/gleason_held"; gl_emb.mkdir(parents=True, exist_ok=True)
    gl_graphs = WORK / "graphs/gleason_held"; gl_graphs.mkdir(parents=True, exist_ok=True)

    FEAT = build_uni_feat()          # raises a clear error if HF_TOKEN / UNI access is missing
    n_ok = 0; skipped = {"no_mask": 0, "no_label": 0, "extract": 0, "graph": 0}
    from tqdm.auto import tqdm
    for fn in tqdm(held_files, desc="gleason held-out cores"):
        stem = Path(fn).stem
        mask_path = find_mask(stem, mask_index)
        if mask_path is None: skipped["no_mask"] += 1; continue
        isup = gleason_core_label(mask_path)
        if isup is None: skipped["no_label"] += 1; continue
        if not extract_gleason_core(FEAT, GLEASON_IMG / fn, stem, gl_emb): skipped["extract"] += 1; continue
        g = build_gleason_graph(stem, isup, gl_emb)
        if g is None: skipped["graph"] += 1; continue
        torch.save(g, gl_graphs / f"{stem}.pt"); n_ok += 1
    print(f"Built {n_ok} held-out Gleason2019 core graphs. Skipped: {skipped}")
    del FEAT; torch.cuda.empty_cache()

    if n_ok >= 5:
        gl_loader = graph_loader(gl_graphs)
        ys, ps, sc = [], [], []
        with torch.no_grad():
            for b in gl_loader:
                b = b.to(DEVICE)
                with torch.cuda.amp.autocast():
                    out = torch.stack([m(b).float() for m in ENS]).mean(0)
                ys += b.y.cpu().tolist(); ps += graph_pred(out).cpu().tolist()
                sc += (out.squeeze(-1).cpu().tolist() if GRAPH_OBJECTIVE == "ordinal" else [])
        qwk, ci = bootstrap_qwk(ys, ps)
        print(f"\nGleason2019 held-out (raw) QWK={qwk:.4f} 95%CI={ci} acc={accuracy_score(ys,ps):.4f} (n={len(ys)})")
        if GRAPH_OBJECTIVE == "ordinal" and OOF_PATH.exists() and sc:
            oof = pd.read_csv(OOF_PATH)
            rounder = OptimizedRounder().fit(oof["score"].values, oof["y_true"].values)
            qwk_cal, ci_cal = bootstrap_qwk(ys, rounder.predict(np.array(sc)))
            print(f"Gleason2019 held-out (calibrated) QWK={qwk_cal:.4f} 95%CI={ci_cal}")
        pd.DataFrame({"y_true": ys, "y_pred": ps}).to_csv(WORK / "gleason_held_results.csv", index=False)
    else:
        print("SKIP evaluation: too few usable held-out core graphs.")

try:
    run_part_a()
except Exception as e:
    # Don't kill Part B / C if only Part A (UNI gated access / token) failed
    print("\n!!! Part A FAILED:", type(e).__name__, "-", str(e)[:400])
    print("    Fix the cause (HF_TOKEN secret attached to this notebook + UNI access) and re-run this cell.")
print("Part A complete.")


Gleason2019 total cores: 244 | held-out (full ~20% split): 45
Mask files indexed: 246 | example image: slide001_core003.jpg | example mask: slide002_core074_classimg_nonconvex.png


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

gleason held-out cores:   0%|          | 0/45 [00:00<?, ?it/s]

Built 45 held-out Gleason2019 core graphs. Skipped: {'no_mask': 0, 'no_label': 0, 'extract': 0, 'graph': 0}

Gleason2019 held-out (raw) QWK=0.8409 95%CI=(0.7478991596638656, 0.9077758094620725) acc=0.6889 (n=45)
Gleason2019 held-out (calibrated) QWK=0.8241 95%CI=(0.7242978276618083, 0.8959149837172903)
Part A complete.


## Part B — Multi-seed / architecture-diverse graph-training ensemble (FULL config)

3 seeds (42/ilra, 123/ilra, 777/mean) x 5-fold x up to 30 epochs -- the full
config, since this notebook has its own separate time budget.

In [4]:

print("=" * 70); print("Part B: Multi-seed + snapshot ensemble (full config)"); print("=" * 70)
SNAPSHOT_LAST_N = 3
RUN_CONFIG = [(42, "ilra"), (123, "ilra"), (777, "mean")]

def train_one_seed_snapshots(seed, pool, slides_df, snapshot_last_n=SNAPSHOT_LAST_N):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    pat = slides_df.groupby("patient")["isup_grade"].agg(lambda s: s.value_counts().index[0]).reset_index()
    _pats, _grades = pat["patient"].values, pat["isup_grade"].values
    _min_class = int(pd.Series(_grades).value_counts().min())
    if len(set(_grades)) > 1 and _min_class >= 2:
        splitter = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(_pats, _grades))
    else:
        splitter = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(_pats))
    kept = []
    for fold, (tr_idx, va_idx) in enumerate(splitter):
        tr_pat = set(pat["patient"].iloc[tr_idx]); va_pat = set(pat["patient"].iloc[va_idx])
        ids_tr = slides_df[slides_df["patient"].isin(tr_pat)]["slide_id"].astype(str).tolist()
        ids_va = slides_df[slides_df["patient"].isin(va_pat)]["slide_id"].astype(str).tolist()
        tl = graph_loader(SICAP_TRAIN_GRAPH_DIR, True, True, ids_tr)
        vl = graph_loader(SICAP_TRAIN_GRAPH_DIR, False, False, ids_va)
        if len(tl.dataset) == 0 or len(vl.dataset) == 0:
            print(f"  seed={seed} pool={pool} fold={fold}: empty split, skip"); continue
        model = GraphTransformerILRA(num_classes=graph_out_dim(), pool=pool).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_GRAPH, eta_min=1e-6)
        scaler = torch.cuda.amp.GradScaler()
        best, bad, snaps = -1, 0, []
        for ep in range(EPOCHS_GRAPH):
            model.train()
            for b in tl:
                b = b.to(DEVICE); opt.zero_grad()
                with torch.cuda.amp.autocast():
                    out = model(b)
                    loss = nn.functional.mse_loss(out.squeeze(-1), b.y.float()) if GRAPH_OBJECTIVE == "ordinal" \
                           else nn.functional.cross_entropy(out, b.y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            sched.step()
            model.eval(); ys, ps = [], []
            with torch.no_grad():
                for b in vl:
                    b = b.to(DEVICE)
                    with torch.cuda.amp.autocast():
                        out = model(b)
                    ys += b.y.cpu().tolist(); ps += graph_pred(out).cpu().tolist()
            qwk = cohen_kappa_score(ys, ps, weights="quadratic", labels=list(range(6))) if len(set(ys)) > 1 else 0.0
            if qwk > best:
                best, bad = qwk, 0
                snaps.append({k: v.cpu().clone() for k, v in model.state_dict().items()})
                snaps = snaps[-snapshot_last_n:]
            else:
                bad += 1
                if bad >= PATIENCE_GRAPH: break
        print(f"  seed={seed} pool={pool} fold={fold}: best_val_qwk={best:.4f}, kept {len(snaps)} snapshot(s)")
        for si, sd in enumerate(snaps):
            kept.append({"seed": seed, "pool": pool, "fold": fold, "snapshot": si, "state_dict": sd})
    return kept

slides_df = pd.read_csv(SICAP_SLIDE_TRAIN)
slides_df["slide_id"] = slides_df["slide_id"].astype(str)
slides_df["patient"] = slides_df["slide_id"].apply(patient_id)

ALL_CKPTS = []
for seed, pool in RUN_CONFIG:
    print(f"\n=== Training seed={seed} pool={pool} ===")
    ALL_CKPTS += train_one_seed_snapshots(seed, pool, slides_df)
print(f"\nTotal checkpoints: {len(ALL_CKPTS)}")

@torch.no_grad()
def predict_with_ckpts(ckpts, loader):
    models = []
    for ck in ckpts:
        m = GraphTransformerILRA(num_classes=graph_out_dim(), pool=ck["pool"]).to(DEVICE)
        m.load_state_dict(ck["state_dict"]); m.eval(); models.append(m)
    ys, scores = [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.cuda.amp.autocast():
            out = torch.stack([m(b).float() for m in models]).mean(0)
        ys += b.y.cpu().tolist()
        scores += (out.squeeze(-1).cpu().tolist() if GRAPH_OBJECTIVE == "ordinal" else [])
    for m in models: del m
    torch.cuda.empty_cache()
    return np.array(ys), np.array(scores)

if panda_gl is not None and len(panda_gl.dataset) and ALL_CKPTS:
    oof = pd.read_csv(OOF_PATH) if OOF_PATH.exists() else None
    rounder = OptimizedRounder().fit(oof["score"].values, oof["y_true"].values) if oof is not None else None

    def eval_cfg(name, ckpts):
        y, sc = predict_with_ckpts(ckpts, panda_gl)
        pred_raw = np.clip(np.round(sc), 0, NUM_ISUP - 1).astype(int)
        qwk_raw, ci_raw = bootstrap_qwk(y, pred_raw)
        row = {"config": name, "n_models": len(ckpts), "QWK_raw": round(qwk_raw, 4), "95CI_raw": ci_raw}
        if rounder is not None:
            qwk_cal, ci_cal = bootstrap_qwk(y, rounder.predict(sc))
            row["QWK_cal"] = round(qwk_cal, 4); row["95CI_cal"] = ci_cal
        print(f"[{name}] n_models={len(ckpts)} QWK_raw={row['QWK_raw']:.4f} QWK_cal={row.get('QWK_cal','-')}")
        return row

    results = []
    for seed, pool in RUN_CONFIG:
        seed_ckpts = [c for c in ALL_CKPTS if c["seed"] == seed]
        if seed_ckpts:
            best_per_fold = {}
            for c in seed_ckpts:
                if c["fold"] not in best_per_fold or c["snapshot"] > best_per_fold[c["fold"]]["snapshot"]:
                    best_per_fold[c["fold"]] = c
            results.append(eval_cfg(f"seed={seed} pool={pool} (best/fold)", list(best_per_fold.values())))
            results.append(eval_cfg(f"seed={seed} pool={pool} +snapshots", seed_ckpts))
    if ALL_CKPTS:
        results.append(eval_cfg("FULL combined ensemble (all seeds+snapshots)", ALL_CKPTS))

    res_df = pd.DataFrame(results)
    res_df.to_csv(WORK / "multiseed_ensemble_results.csv", index=False)
    print("\n", res_df.to_string(index=False))
    single_qwks = [r["QWK_raw"] for r in results if "(best/fold)" in r["config"]]
    if len(single_qwks) >= 2:
        print(f"\nAcross {len(single_qwks)} seeds: mean={np.mean(single_qwks):.4f} std={np.std(single_qwks):.4f}")
else:
    print("SKIP Part B eval: PANDA graphs unavailable or no checkpoints collected.")
print("Part B complete.")


Part B: Multi-seed + snapshot ensemble (full config)

=== Training seed=42 pool=ilra ===
  seed=42 pool=ilra fold=0: best_val_qwk=0.8994, kept 3 snapshot(s)
  seed=42 pool=ilra fold=1: best_val_qwk=0.8577, kept 3 snapshot(s)
  seed=42 pool=ilra fold=2: best_val_qwk=0.8413, kept 3 snapshot(s)
  seed=42 pool=ilra fold=3: best_val_qwk=0.8599, kept 3 snapshot(s)
  seed=42 pool=ilra fold=4: best_val_qwk=0.8794, kept 3 snapshot(s)

=== Training seed=123 pool=ilra ===
  seed=123 pool=ilra fold=0: best_val_qwk=0.8713, kept 3 snapshot(s)
  seed=123 pool=ilra fold=1: best_val_qwk=0.8561, kept 3 snapshot(s)
  seed=123 pool=ilra fold=2: best_val_qwk=0.8916, kept 3 snapshot(s)
  seed=123 pool=ilra fold=3: best_val_qwk=0.8998, kept 3 snapshot(s)
  seed=123 pool=ilra fold=4: best_val_qwk=0.9027, kept 3 snapshot(s)

=== Training seed=777 pool=mean ===
  seed=777 pool=mean fold=0: best_val_qwk=0.8341, kept 3 snapshot(s)
  seed=777 pool=mean fold=1: best_val_qwk=0.8684, kept 3 snapshot(s)
  seed=777 poo

## Part C — Interpretability & robustness analyses

Uses Part 1's fold ensemble (`ENS`) + cached embeddings/graphs. No retraining.

In [5]:

print("--- Analysis 1: attention-to-annotation concordance ---")
try:
    val_labels_df = pd.read_csv(SICAP_VAL_LABELS)
    val_labels_df["slide_id"] = val_labels_df["filename"].apply(get_slide_id)
    val_labels_df["coord"] = val_labels_df["filename"].apply(get_coords)

    @torch.no_grad()
    def slide_attention_10x(model, data):
        data = data.clone().to(DEVICE)
        data.batch = torch.zeros(data.x.shape[0], dtype=torch.long, device=DEVICE)
        _ = model(data)
        importance = model.ilra.last_attn[0].mean(0).cpu().numpy()
        return importance[:data.x.shape[0] // 2]

    conc_rows = []
    for gf in sorted(SICAP_VAL_GRAPH_DIR.glob("*.pt")):
        sid = gf.stem
        slide_rows = val_labels_df[val_labels_df["slide_id"] == sid]
        if slide_rows.empty: continue
        coord2label = {row["coord"]: row["binary_label"] for _, row in slide_rows.iterrows()}
        coords_path = INPUT_ROOT / "embeddings/sicap/val" / f"{sid}_coords.npy"
        if not coords_path.exists(): continue
        coords = np.load(coords_path)
        data = torch.load(gf, weights_only=False)
        N = min(len(coords), data.x.shape[0] // 2)
        if N < 3: continue
        importance = slide_attention_10x(ENS[0], data)[:N]; coords = coords[:N]
        gt = np.array([coord2label.get((int(r), int(c)), np.nan) for r, c in coords])
        valid = ~np.isnan(gt)
        if valid.sum() < 3: continue
        importance, gt = importance[valid], gt[valid].astype(int)
        gt_mask = gt == 1
        if gt_mask.sum() == 0 or gt_mask.sum() == len(gt_mask): continue
        k = max(1, int(round(0.30 * len(importance))))
        pred_mask = np.zeros(len(importance), dtype=bool); pred_mask[np.argsort(importance)[-k:]] = True
        inter = (pred_mask & gt_mask).sum(); union = (pred_mask | gt_mask).sum()
        conc_rows.append({"slide_id": sid, "IoU": inter / union if union > 0 else np.nan,
                          "Dice": 2 * inter / (pred_mask.sum() + gt_mask.sum())})
    conc_df = pd.DataFrame(conc_rows)
    conc_df.to_csv(WORK / "attention_annotation_concordance.csv", index=False)
    print(f"Scored {len(conc_df)} slides. Mean IoU={conc_df['IoU'].mean():.3f} Mean Dice={conc_df['Dice'].mean():.3f}")
except Exception as e:
    print("Analysis 1 failed/skipped:", e)

print("\n--- Analysis 2: embedding t-SNE/UMAP ---")
try:
    def mean_pool_embeddings(emb_dir, slide_csv, tag, limit=400):
        df = pd.read_csv(slide_csv)
        if limit and len(df) > limit: df = df.sample(limit, random_state=SEED)
        rows = []
        for _, r in df.iterrows():
            sid = str(r["slide_id"]); isup = int(r["isup_grade"])
            f = Path(emb_dir) / f"{sid}_10x.npy"
            if not f.exists(): continue
            e = np.load(f)
            if len(e) == 0: continue
            rows.append({"slide_id": sid, "isup_grade": isup, "dataset": tag, "vec": e.mean(0)})
        return rows

    sicap_rows = mean_pool_embeddings(INPUT_ROOT / "embeddings/sicap/val", SICAP_SLIDE_VAL, "SICAP")
    panda_rows = mean_pool_embeddings(INPUT_ROOT / "embeddings/panda", PANDA_SLIDE_LABELS, "PANDA") \
                 if PANDA_SLIDE_LABELS.exists() else []
    emb_df = pd.DataFrame(sicap_rows + panda_rows)
    X = np.stack(emb_df["vec"].values)
    perp = min(30, max(5, len(X) // 4 - 1))
    e2d = TSNE(n_components=2, random_state=SEED, perplexity=perp).fit_transform(X)
    emb_df["tsne_x"], emb_df["tsne_y"] = e2d[:, 0], e2d[:, 1]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ds, c in [("SICAP", "tab:blue"), ("PANDA", "tab:orange")]:
        m = emb_df["dataset"] == ds
        axes[0].scatter(emb_df.loc[m, "tsne_x"], emb_df.loc[m, "tsne_y"], s=14, alpha=0.6, label=ds, c=c)
    axes[0].set_title("t-SNE by dataset (domain gap)"); axes[0].legend()
    sca = axes[1].scatter(emb_df["tsne_x"], emb_df["tsne_y"], s=14, c=emb_df["isup_grade"], cmap="viridis")
    axes[1].set_title("t-SNE by ISUP grade"); plt.colorbar(sca, ax=axes[1])
    plt.tight_layout(); plt.savefig(FIG_DIR / "tsne_domain_grade.png", dpi=150); plt.close()
    emb_df.drop(columns=["vec"]).to_csv(WORK / "embedding_projection.csv", index=False)
    print(f"t-SNE saved ({len(sicap_rows)} SICAP + {len(panda_rows)} PANDA slides).")
except Exception as e:
    print("Analysis 2 failed/skipped:", e)

print("\n--- Analysis 3: uncertainty vs error ---")
try:
    @torch.no_grad()
    def per_fold_predictions(models, loader):
        out = {}
        for m in models:
            m.eval()
            for b in loader:
                b = b.to(DEVICE)
                with torch.cuda.amp.autocast():
                    o = m(b)
                preds = graph_pred(o).cpu().tolist(); trues = b.y.cpu().tolist()
                sids = b.slide_id if isinstance(b.slide_id, (list, tuple)) else [b.slide_id]
                for sid, t, p in zip(sids, trues, preds):
                    sid = str(sid); out.setdefault(sid, {"y_true": t, "preds": []}); out[sid]["preds"].append(p)
        return out

    if panda_gl is not None and len(panda_gl.dataset):
        fold_preds = per_fold_predictions(ENS, panda_gl)
        unc_rows = [{"slide_id": sid, "y_true": d["y_true"], "fold_std": float(np.std(d["preds"])),
                    "abs_error": abs(int(round(np.mean(d["preds"]))) - d["y_true"])}
                    for sid, d in fold_preds.items()]
        unc_df = pd.DataFrame(unc_rows); unc_df.to_csv(WORK / "panda_uncertainty.csv", index=False)
        corr = unc_df["fold_std"].corr(unc_df["abs_error"])
        print(f"Correlation(fold_std, abs_error) = {corr:.3f}")
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(unc_df["fold_std"], unc_df["abs_error"], alpha=0.5, s=20)
        ax.set_xlabel("Fold disagreement (std)"); ax.set_ylabel("Absolute error")
        ax.set_title(f"PANDA: uncertainty vs error (r={corr:.2f})")
        plt.tight_layout(); plt.savefig(FIG_DIR / "uncertainty_vs_error.png", dpi=150); plt.close()
    else:
        print("SKIP: PANDA graphs unavailable.")
except Exception as e:
    print("Analysis 3 failed/skipped:", e)

print("\n--- Analysis 5: error severity breakdown ---")
try:
    def error_breakdown(y_true, y_pred, name):
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        diff = np.abs(y_true - y_pred)
        row = {"dataset": name, "n": len(y_true), "exact_%": (diff == 0).mean() * 100,
               "adjacent_%": (diff == 1).mean() * 100, "moderate_%": (diff == 2).mean() * 100,
               "major_%": (diff >= 3).mean() * 100}
        print(f"[{name}] exact={row['exact_%']:.1f}% adjacent={row['adjacent_%']:.1f}% "
              f"moderate={row['moderate_%']:.1f}% major={row['major_%']:.1f}%")
        return row

    val_loader = graph_loader(SICAP_VAL_GRAPH_DIR)
    yv, pv = [], []
    with torch.no_grad():
        for b in val_loader:
            b = b.to(DEVICE)
            with torch.cuda.amp.autocast():
                out = torch.stack([m(b).float() for m in ENS]).mean(0)
            yv += b.y.cpu().tolist(); pv += graph_pred(out).cpu().tolist()
    rows = [error_breakdown(yv, pv, "SICAP val")]
    if panda_gl is not None and len(panda_gl.dataset):
        yp, pp = [], []
        with torch.no_grad():
            for b in panda_gl:
                b = b.to(DEVICE)
                with torch.cuda.amp.autocast():
                    out = torch.stack([m(b).float() for m in ENS]).mean(0)
                yp += b.y.cpu().tolist(); pp += graph_pred(out).cpu().tolist()
        rows.append(error_breakdown(yp, pp, "PANDA"))
    pd.DataFrame(rows).to_csv(WORK / "error_breakdown.csv", index=False)
except Exception as e:
    print("Analysis 5 failed/skipped:", e)

print("\nPart C complete. All outputs saved under /kaggle/working/.")


--- Analysis 1: attention-to-annotation concordance ---
Analysis 1 failed/skipped: 'IoU'

--- Analysis 2: embedding t-SNE/UMAP ---
t-SNE saved (33 SICAP + 300 PANDA slides).

--- Analysis 3: uncertainty vs error ---
Correlation(fold_std, abs_error) = 0.283

--- Analysis 5: error severity breakdown ---
[SICAP val] exact=66.7% adjacent=30.3% moderate=3.0% major=0.0%
[PANDA] exact=39.3% adjacent=39.0% moderate=11.0% major=10.7%

Part C complete. All outputs saved under /kaggle/working/.


## Summary

Outputs saved under `/kaggle/working/`:
- `gleason_held_results.csv` -- Part A
- `multiseed_ensemble_results.csv` -- Part B (report the "FULL combined ensemble"
  row as the headline number, and the per-seed mean +/- SD as stability evidence)
- `attention_annotation_concordance.csv`, `embedding_projection.csv`,
  `panda_uncertainty.csv`, `error_breakdown.csv`, `analysis_figures/*.png` -- Part C

Share these back for interpretation and for finalising the numbers in the paper.
